# HCMAIC P3 keyframe runner

Notebook chạy **đầy đủ P3** bằng chính các module backend: TransNetV2 → repair/split shot → adaptive candidate + quality fallback → MobileCLIP2 → K-medoids/AUCC → common/unique/global MMR → dedup. Thuật toán không bị copy thành bản thứ hai nên local và Kaggle luôn dùng cùng implementation.

Trước khi chạy: bật **GPU** và **Internet**, attach dataset video cùng file weight TransNetV2, và bảo đảm branch cấu hình bên dưới đã được push.

In [ ]:
REPO_URL = "https://github.com/iamtro-2006/event-retrieval-system.git"
BRANCH = "UpdateKeyframeP3"
GROUPS = ["Videos_L21_a", "Videos_L22_a"]
VIDEO_IDS = []  # Rỗng = chạy toàn bộ video trong các GROUPS ở trên
GROUP_PATHS = {}  # Tùy chọn khi trùng tên, ví dụ {"Videos_L21_a": "/kaggle/input/my-data/Videos_L21_a"}
TRANSNET_WEIGHTS_PATH = None  # Tùy chọn; để None để tự tìm trong /kaggle/input

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

working = Path("/kaggle/working")
repo = working / "event-retrieval-system"
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "open_clip_torch==3.3.0", "decord==0.6.0", "ffmpeg-python==0.2.0", "scikit-learn==1.7.2", "PyYAML>=6.0"], check=True)

In [ ]:
import cv2
import open_clip
import torch
import yaml

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. In Kaggle Settings, select a GPU accelerator and restart the session.")
if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg executable is unavailable in this Kaggle image.")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| OpenCV:", cv2.__version__, "| OpenCLIP:", open_clip.__version__)

In [ ]:
input_root = Path("/kaggle/input")
video_extensions = {".mp4", ".mkv", ".avi", ".mov", ".webm"}
if not GROUPS:
    raise ValueError("GROUPS must contain at least one Videos_* folder.")

def video_files(directory):
    return sorted(path for path in directory.rglob("*") if path.is_file() and path.suffix.lower() in video_extensions)

def resolve_group(group):
    if group in GROUP_PATHS:
        selected = Path(GROUP_PATHS[group])
        if not selected.is_dir() or not video_files(selected):
            raise FileNotFoundError(f"GROUP_PATHS[{group!r}] is missing or contains no videos: {selected}")
        return selected.resolve()
    hits = [path for path in input_root.rglob(group) if path.is_dir() and video_files(path)]
    if len(hits) != 1:
        raise RuntimeError(f"Expected exactly one video folder named {group!r}; found {hits}. Set GROUP_PATHS to disambiguate.")
    return hits[0].resolve()

resolved_groups = {group: resolve_group(group) for group in GROUPS}
dataset_root = working / "p3_video_input"
if dataset_root.exists():
    shutil.rmtree(dataset_root)
dataset_root.mkdir(parents=True)
for group, source in resolved_groups.items():
    videos = video_files(source)
    for video in videos:
        link = dataset_root / group / video.relative_to(source)
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(video)
    print(f"{group}: {len(videos)} video(s) <- {source}")

if TRANSNET_WEIGHTS_PATH:
    weights_path = Path(TRANSNET_WEIGHTS_PATH)
    if not weights_path.is_file():
        raise FileNotFoundError(f"TransNetV2 weights not found: {weights_path}")
else:
    weight_hits = list(input_root.rglob("transnetv2-pytorch-weights.pth"))
    if not weight_hits:
        weight_hits = list(input_root.rglob("*transnet*.pth"))
    if len(weight_hits) != 1:
        raise RuntimeError(f"Expected exactly one TransNetV2 weight file; found {weight_hits}. Set TRANSNET_WEIGHTS_PATH.")
    weights_path = weight_hits[0]
print("Unified input view:", dataset_root)
print("TransNet weights:", weights_path)

In [ ]:
backend = repo / "backend"
source_config = backend / "configs" / "kf_extraction.yaml"
runtime_config = working / "kf_extraction_kaggle.yaml"
with source_config.open(encoding="utf-8") as stream:
    config = yaml.safe_load(stream)
config["paths"]["input_dir"] = str(dataset_root)
config["paths"]["output_dir"] = str(working / "processed")
config["paths"]["cache_dir"] = str(working / ".cache" / "keyframe_extraction")
config["transnet"]["weights_path"] = str(weights_path)
config["transnet"]["device"] = "cuda"
config["keyframe"]["selection_embedding"]["device"] = "cuda"
config["keyframe"]["write_diagnostics"] = False
config["logging"]["log_to_file"] = False
expected_p3 = {
    ("keyframe", "strategy"): "p3",
    ("transnet", "batch_size"): 500,
    ("keyframe", "selection_embedding", "model_name"): "MobileCLIP2-S4",
    ("keyframe", "selection_embedding", "pretrained"): "dfndr2b",
    ("keyframe", "candidate", "min_gap_sec"): 0.25,
    ("keyframe", "candidate", "max_gap_sec"): 1.0,
    ("keyframe", "clustering", "max_clusters_per_shot"): 5,
    ("keyframe", "selector", "mmr_lambda"): 0.60,
    ("keyframe", "dedup", "dense_cosine_threshold"): 0.97,
}
for keys, expected in expected_p3.items():
    value = config
    for key in keys:
        value = value[key]
    if value != expected:
        raise RuntimeError(f"P3 config mismatch at {'.'.join(keys)}: expected {expected!r}, got {value!r}")
sys.path.insert(0, str(backend))
from src.keyframe_extraction.models.candidate_sampler import sample_candidates
from src.keyframe_extraction.models.clustering import cluster_candidates
from src.keyframe_extraction.models.deduplicator import deduplicate
from src.keyframe_extraction.models.detector import detect_scenes, repair_and_split_scenes
from src.keyframe_extraction.models.p3_selector import select_p3
from src.keyframe_extraction.models.quality_filter import evaluate_quality
print("P3 modules OK: TransNet, repair/split, candidate+quality fallback, MobileCLIP2, K-medoids/AUCC, common+unique+MMR, dedup")
with runtime_config.open("w", encoding="utf-8") as stream:
    yaml.safe_dump(config, stream, sort_keys=False)

command = [sys.executable, "scripts/keyframe_extraction/run.py", "--config", str(runtime_config)]
if GROUPS:
    command += ["--groups", *GROUPS]
if VIDEO_IDS:
    command += ["--video-ids", *VIDEO_IDS]
subprocess.run(command, cwd=backend, check=True)

In [ ]:
output = Path("/kaggle/working/processed")
images = list((output / "keyframes").rglob("*.jpg"))
maps = list((output / "map_keyframes").rglob("*.csv"))
print(f"Completed: {len(images)} keyframe images, {len(maps)} mapping files")
print("Kaggle output:", output)